# 🎓 Student Engagement Detection — Temporal Transformer DAiSEE Transfer Learning

This experiment implements **Transfer Learning on the Temporal Transformer Head** (from `colab_transformer`) using the public **DAiSEE** dataset:

1. **Phase 1 (Pre-training):** Pretrain the **Temporal Transformer Encoder** (`d_model=128`, `nhead=4`, `num_layers=2`) on the public DAiSEE dataset to learn generalized student temporal dynamics.
2. **Phase 2 (Fine-tuning):** Transfer temporal Transformer weights to your custom ~100 clips dataset, adapt the top classification layer to 3 classes (`Engaged`, `Partially Engaged`, `Disengaged`), and fine-tune.

### Step 1: Mount Google Drive & Set Directory

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Student_Engagement_DL"
TRANSFER_DIR = os.path.join(PROJECT_ROOT, "colab_daisee_transfer")

os.chdir(TRANSFER_DIR)
sys.path.append(TRANSFER_DIR)
print("Current Working Directory:", os.getcwd())

### Step 2: Install Dependencies & Check Paths

In [ ]:
!pip install -q torch torchvision pandas scikit-learn matplotlib seaborn tqdm

import config_daisee

print("--- PATH & DATASET STATUS ---")
print(f"DAiSEE Zip Path Detected:  {config_daisee.DAISEE_ZIP_PATH}")
print(f"Target Fused Features Dir: {config_daisee.TARGET_FUSED_DIR}")

### Step 3: Run Pipeline Verification Test

In [ ]:
# Verify 2-Phase Temporal Transformer Transfer Learning workflow
!python test_pipeline_transfer.py

### Step 4: Phase 1 — Pre-train Temporal Transformer Head on DAiSEE

In [ ]:
# Pre-train Temporal Transformer Encoder model on DAiSEE
!python pretrain_daisee.py --model transformer --epochs 30 --lr 0.0005

### Step 5: Phase 2 — Fine-Tune Pretrained Temporal Transformer on Target Dataset (~100 Clips)

In [ ]:
# Load pre-trained DAiSEE Transformer weights and fine-tune on your 100 clips dataset (3 classes)
!python finetune_target.py --model transformer --epochs 40 --lr 0.0001

### Step 6: Evaluate Fine-Tuned Model & View Results

In [ ]:
!python evaluate_transfer.py --model transformer

### Step 7: Display Confusion Matrix & Transfer Learning Metrics

In [ ]:
from IPython.display import Image, display
import json

metrics_path = "evaluation_results_daisee_transfer/metrics_daisee_transfer.json"
cm_path = "evaluation_results_daisee_transfer/confusion_matrix_daisee_transfer.png"

if os.path.exists(metrics_path):
    print("\n==================== DAISEE TRANSFORMER TRANSFER RESULTS ====================")
    with open(metrics_path, "r") as f:
        data = json.load(f)
    print(f"Accuracy:  {data.get('accuracy', 0)*100:.2f}%")
    print(f"Precision: {data.get('precision_weighted', 0)*100:.2f}%")
    print(f"Recall:    {data.get('recall_weighted', 0)*100:.2f}%")
    print(f"F1-Score:  {data.get('f1_weighted', 0)*100:.2f}%")
    
if os.path.exists(cm_path):
    display(Image(cm_path))